# Spectroscopy: CCD Characterisation and master dark

`author` Julien Morin <julien.morin@umontpellier.fr>

`date` 18 Oct 2023

Reference: *Handbook of CCD Astronomy*, §3.5
* compute master dark
* identify hot pixels
* compute dark current

In [ ]:
# import and settings
%matplotlib notebook
import copy
import os
import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage
from astropy.stats import mad_std
from astropy import units as u
from astropy.visualization import ZScaleInterval
import ccdproc as ccdp

## Load previously computed data 

In [ ]:
# define CCD characteristics
ccd_gain = 0.37 # e-/ADU
ccd_ron = 6.0 # e-

In [ ]:
# load master bias and print stats
master_bias = ccdp.CCDData.read('reference/master_bias.fits', unit='adu')

npix = master_bias.shape[0]*master_bias.shape[1]
print('npix    mean    median  std  ')
print('-----------------------------')
print("%7d %5.2f   %5.2f  %5.2f" % (npix, np.mean(master_bias.data), np.median(master_bias.data), np.std(master_bias.data)))

## Master dark

In [ ]:
# create dark frame collection and print properties
ifc_dark30 = ccdp.ImageFileCollection('calib/', glob_include='Dark_30s*.fit*')
print(ifc_dark30.summary['file', 'date-obs', 'objname', 'exptime'])

In [ ]:
# compute master dark 30s
master_dark30 = ccdp.combine(ifc_dark30.files_filtered(include_path=True), unit='adu', combine='average', mem_limit=512.e6)

In [ ]:
# set header keyword (will appear in the saved file)
master_dark30.meta['COMBINED'] = True

In [ ]:
### Further work: compare statiscal properties of the series and the master frame

In [ ]:
# display 1st dark, master dark and residual
fig = plt.figure(figsize=(8,12))
#
ax1 = fig.add_subplot(311)
ccd_dark1 = ccdp.CCDData.read(ifc_dark30.location+ifc_dark30.files[0], unit='adu')
interval1 = ZScaleInterval()
z1, z2 = interval1.get_limits(ccd_dark1)
im1 = ax1.imshow(ccd_dark1.data, origin='lower', vmin=z1, vmax=z2)
fig.colorbar(im1)
ax1.set_title(ifc_dark30.files[0])
#
ax2 = fig.add_subplot(312)
im2 = ax2.imshow(master_dark30.data, origin='lower', vmin=z1, vmax=z2)
fig.colorbar(im2)
ax2.set_title('master dark 30s')
#
ax3 = fig.add_subplot(313)
res_dark = ccd_dark1.data - master_dark30.data
interval2 = ZScaleInterval()
z1, z2 = interval2.get_limits(res_dark)
im3 = ax3.imshow(res_dark, origin='lower', vmin=z1, vmax=z2)
fig.colorbar(im3)
ax3.set_title('dark 30s residual')
plt.show()

In [ ]:
# Further work: print statistics of the residual image

# Hot pixels

In [ ]:
# bias subtract the master dark
master_dark30_b = ccdp.subtract_bias(master_dark30, master_bias)


In [ ]:
### Further work: print stats of the bias-subtracted master dark

In [ ]:
### Further work: identify hot pixels (hp)

In [ ]:
### Further work: propose ideas to correct hot pixels on the master dark

In [ ]:
### Further work: compare stats before/after hot pixels clean + compare images

## Dark current

In [ ]:
### Further work: compute dark current from the hot pixel-corrected dark frame

**NOTE:** the Atik Infinity camera is not thermally-regulated, hence we get inconsistent results between 30s and 60s exposures

## Save master dark and hot pixel mask

In [ ]:
master_dark30_bh = ccdp.CCDData(data=master_dark30_b.data, meta=master_dark30_b.meta, unit='adu')
master_dark30_b.write('reference/master_dark30.fits', overwrite=True)
### Further work: save the hp-corrected dark + associated hp mask
#master_dark30_bh = ccdp.CCDData(data=master_dark30_bh.data, meta=master_dark30_bh.meta, mask=hot_pixels_30, unit='adu')
#master_dark30_b.write('reference/master_dark30.fits', overwrite=True)